# Harry Potter RAG — Lab 1

A brief example of a simple workflow of a RAG pipeline using the famous *Harry Potter & The Philosopher's Stone* by J.K. Rowling, including:

- Text processing
- Chunking
- Embedding
- VectorDB management
- Q&A
- Evaluation

*Disclaimer: The source book is copyrighted and is not included in this repository. This lab is for personal, non-commercial study only — only original code, questions, and metrics are published. AI coding agent is used in the process to assist notes writing and project planning.*

## Import dependencies

In [31]:
import boto3
from botocore.config import Config
import pymupdf
import re
import json
import os
import hashlib
from pathlib import Path
import tiktoken
import numpy as np
from tqdm.auto import tqdm
import chromadb
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi

In [3]:
def repo_root() -> Path:
    """Walk up from the cwd until the repo root (the dir containing .git)."""
    p = Path.cwd()
    while p != p.parent and not (p / ".git").exists():
        p = p.parent
    return p

ROOT = repo_root()
LABS = ROOT / "labs"
DATA = LABS / "data"

load_dotenv(dotenv_path=str(LABS / ".env"))
os.getenv("AWS_REGION")


'ap-northeast-1'

## Setup Bedrock for LLM

Consist of 2 functions:

1. Conversation: Use it for bedrock health check, chat function, Q&A for RAG as well as evaluation using LLM-as-a-judge
2. Embedding: text embedding of Harry Potter's content for RAG

In [4]:
class Bedrock:
    """Bedrock helpers: Converse for chat, Titan for embeddings (sequential + cached)."""

    def __init__(self, model_id=None, region=None, embedding_model_id=None):
        self.region = region or os.getenv("AWS_REGION") or "ap-northeast-1"
        self.model_id = model_id or os.getenv("BEDROCK_CONVERSATION_MODEL_ID")
        if not self.model_id:
            raise ValueError("Set BEDROCK_CONVERSATION_MODEL_ID in .env")
        self.embedding_model_id = embedding_model_id or os.getenv("BEDROCK_EMBEDDING_MODEL_ID")
        if not self.embedding_model_id:
            raise ValueError("Set BEDROCK_EMBEDDING_MODEL_ID in .env")
        self.client = boto3.client(
            "bedrock-runtime",
            region_name=self.region,
            config=Config(read_timeout=120, retries={"max_attempts": 5, "mode": "standard"}),
        )

    # ---- conversation (Q&A and judge): override model/params per call ----
    def chat(self, prompt, system=None, model_id=None, temperature=0.0, max_tokens=1024):
        kwargs = {
            "modelId": model_id or self.model_id,
            "messages": [{"role": "user", "content": [{"text": prompt}]}],
            "inferenceConfig": {"temperature": temperature, "maxTokens": max_tokens},
        }
        if system:
            kwargs["system"] = [{"text": system}]
        response = self.client.converse(**kwargs)
        return response["output"]["message"]["content"][0]["text"]

    # ---- embeddings: sequential + disk cache ----
    @staticmethod
    def _cache_path():
        return DATA / "embeddings_cache.jsonl"

    @staticmethod
    def _cache_key(text, model_id, dimensions, normalize):
        raw = f"{model_id}|{dimensions}|{normalize}|{text}"
        return hashlib.sha256(raw.encode("utf-8")).hexdigest()

    def _embed_one(self, text, dimensions, normalize):
        body = json.dumps({"inputText": text, "dimensions": dimensions, "normalize": normalize})
        resp = self.client.invoke_model(modelId=self.embedding_model_id, body=body)
        return json.loads(resp["body"].read())["embedding"]

    def embed(self, texts, dimensions=1024, normalize=True, cache_path=None):
        """Embed texts one by one, appending results to a JSONL cache.

        Returns an (n, dimensions) float32 array. 
        Re-running the notebook will proceed search for already-cached texts, 
        those will make no API calls.
        """
        if isinstance(texts, str):
            texts = [texts]
        texts = list(texts)
        cache_path = Path(cache_path) if cache_path else self._cache_path()

        cache = {}
        if cache_path.exists():
            for line in cache_path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    rec = json.loads(line)
                    cache[rec["key"]] = rec["vector"]

        model = self.embedding_model_id
        keys = [self._cache_key(t, model, dimensions, normalize) for t in texts]
        todo = [(i, t) for i, t in enumerate(texts) if keys[i] not in cache]

        if todo:
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            with open(cache_path, "a", encoding="utf-8") as f:

                # Process bar
                for i, t in tqdm(todo, desc="embedding"):
                    vec = self._embed_one(t, dimensions, normalize)
                    cache[keys[i]] = vec
                    f.write(json.dumps({"key": keys[i], "vector": vec}) + "\n")

        return np.array([cache[k] for k in keys], dtype=np.float32)


bedrock_client = Bedrock()
judge_client = Bedrock(model_id=os.getenv("BEDROCK_JUDGE_MODEL_ID"))


## Import data (the book)

From `get_toc()` we know the pdf is structured with chapter+1 as the `pymupdf` index number

In [5]:
doc = pymupdf.open(str(DATA / "harrypotter.pdf"))
toc = doc.get_toc()
toc[:25]


[[1, 'Title Page', 2],
 [1, 'Pottermore.com', 4],
 [1, 'Contents', 6],
 [1, 'Harry Potter and the Sorcerer’s Stone', 7],
 [2, 'Chapter 1 - The Boy Who Lived', 12],
 [2, 'Chapter 2 - The Vanishing Glass', 26],
 [2, 'Chapter 3 - The Letters from No One', 37],
 [2, 'Chapter 4 - The Keeper of the Keys', 50],
 [2, 'Chapter 5 - Diagon Alley', 62],
 [2, 'Chapter 6 - The Journey from Platform Nine and Three-quarters', 85],
 [2, 'Chapter 7 - The Sorting Hat', 107],
 [2, 'Chapter 8 - The Potions Master', 123],
 [2, 'Chapter 9 - The Midnight Duel', 133],
 [2, 'Chapter 10 - Halloween', 150],
 [2, 'Chapter 11 - Quidditch', 164],
 [2, 'Chapter 12 - The Mirror of Erised', 176],
 [2, 'Chapter 13 - Nicolas Flamel', 194],
 [2, 'Chapter 14 - Norbert the Norwegian Ridgeback', 205],
 [2, 'Chapter 15 - The Forbidden Forest', 217],
 [2, 'Chapter 16 - Through the Trapdoor', 234],
 [2, 'Chapter 17 - The Man with Two Faces', 257],
 [2, 'Copyright page', 275],
 [1, 'Harry Potter and the Chamber of Secrets', 276]

TOC contains copyright and other extra pages not required in RAG building.

Example of actual book content. Requires cleaning of `\n` separator as well as joining pages to avoid breaking words / context.

In [6]:
doc[12-1].get_text()[:250]

'M\n \nCHAPTER  ONE\nTHE BOY WHO LIVED\nr. and Mrs. Dursley, of number four, Privet Drive, were proud to say\nthat they were perfectly normal, thank you very much. They were the\nlast people you’d expect to be involved in anything strange or mysterious,\nbec'

## Building index / metadata

In [7]:
# Get book names
book_names = [
    chapter[1] for chapter in toc if chapter[1].startswith("Harry Potter")
]

In [8]:
_CHAPTER_RE = re.compile(r"^Chapter\s+(\d+)\s*-\s*(.+)$", re.IGNORECASE)
_CONTENT_SPECIAL = {"Prologue", "Epilogue"}
_BACK_MATTER = "Copyright page"

book_names = [t[1] for t in toc if t[1].startswith("Harry Potter")]
index = []
book_num = 0

def _add_section(book_num, chapter_num, name, page)->None:
    index.append({
        "book_num": book_num,
        "book_name": book_names[book_num - 1],
        "chapter_num": chapter_num,      # None for Prologue/Epilogue
        "chapter_name": name,
        "chapter_start_page_num": page,
        "chapter_end_page_num": None,
    })

for level, title, page in toc:
    title = title.strip()

    if title in book_names:              # book title row -> just bump the counter
        book_num += 1
        continue

    if title == _BACK_MATTER:            # back matter CAPS the previous section
        if index and index[-1]["chapter_end_page_num"] is None:
            index[-1]["chapter_end_page_num"] = page - 1
        continue

    if title in _CONTENT_SPECIAL:
        _add_section(book_num, None, title, page)
        continue

    m = _CHAPTER_RE.match(title)
    if m:
        _add_section(book_num, int(m.group(1)), title, page)

# any section not already capped by back matter ends before the next content section
for prev, nxt in zip(index, index[1:]):
    if prev["chapter_end_page_num"] is None:
        prev["chapter_end_page_num"] = nxt["chapter_start_page_num"] - 1

# final section (book 7's Epilogue) if no Copyright follows it
if index and index[-1]["chapter_end_page_num"] is None:
    index[-1]["chapter_end_page_num"] = doc.page_count

index[:5]

[{'book_num': 1,
  'book_name': 'Harry Potter and the Sorcerer’s Stone',
  'chapter_num': 1,
  'chapter_name': 'Chapter 1 - The Boy Who Lived',
  'chapter_start_page_num': 12,
  'chapter_end_page_num': 25},
 {'book_num': 1,
  'book_name': 'Harry Potter and the Sorcerer’s Stone',
  'chapter_num': 2,
  'chapter_name': 'Chapter 2 - The Vanishing Glass',
  'chapter_start_page_num': 26,
  'chapter_end_page_num': 36},
 {'book_num': 1,
  'book_name': 'Harry Potter and the Sorcerer’s Stone',
  'chapter_num': 3,
  'chapter_name': 'Chapter 3 - The Letters from No One',
  'chapter_start_page_num': 37,
  'chapter_end_page_num': 49},
 {'book_num': 1,
  'book_name': 'Harry Potter and the Sorcerer’s Stone',
  'chapter_num': 4,
  'chapter_name': 'Chapter 4 - The Keeper of the Keys',
  'chapter_start_page_num': 50,
  'chapter_end_page_num': 61},
 {'book_num': 1,
  'book_name': 'Harry Potter and the Sorcerer’s Stone',
  'chapter_num': 5,
  'chapter_name': 'Chapter 5 - Diagon Alley',
  'chapter_start_pag

## Get text by chapter

In [9]:
def get_text_from_chapter(chapter)->str:
    start = chapter["chapter_start_page_num"] - 1    # PymuPDF is 1-based on page index
    end = chapter["chapter_end_page_num"]          

    lines = []
    for page in doc[start:end]:
        lines.extend(page.get_text().split("\n"))

    text = ""
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if text and not text.endswith("-"):        # wrapped hyphen special case
            text += " "
        text += line

    return re.sub(r"\s+", " ", text).strip()       # collapse any leftover runs
        
text_data = []

for chapter in index:
    if chapter['book_num'] == 1:
        text_data.append(get_text_from_chapter(chapter))


In [10]:
len(text_data)

17

In [11]:
with open(DATA / "harrypotter-rawtext.json", "w", encoding="utf-8") as f:
    json.dump(text_data, f)


## Chunking text by chapter - Naive Chunking

`tiktoken` is a useful library to split text by token size. LangChain example:  https://docs.langchain.com/oss/python/integrations/splitters/split_by_token

`tiktoken` doc: https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb

In [12]:
encoding = tiktoken.get_encoding("o200k_base")
len(encoding.encode(text_data[0]))

6327

In [13]:
# Check token/text count per chapter
list(zip(
    [i for i in range(1,18)],
    [len(encoding.encode(text)) for text in text_data],
    [len(text.split()) for text in text_data]
))

[(1, 6327, 4727),
 (2, 4475, 3505),
 (3, 5016, 3924),
 (4, 5246, 3813),
 (5, 9279, 6717),
 (6, 8518, 6387),
 (7, 6033, 4537),
 (8, 3962, 3071),
 (9, 6836, 4989),
 (10, 5651, 4322),
 (11, 4641, 3400),
 (12, 7215, 5569),
 (13, 4420, 3245),
 (14, 4795, 3535),
 (15, 7085, 5187),
 (16, 8862, 6611),
 (17, 7764, 5756)]

Function to chunk text by sliding window with overlaps

In [14]:
def chunk_text_by_sliding_window(
    text,
    chunk_token_size=512,
    overlap_ratio=0.15,
) -> list[dict]:
    # Guard bad params (a step of 0 would loop forever).
    if not 0 <= overlap_ratio < 1:
        raise ValueError("overlap_ratio must be in [0, 1)")

    encoded_text = encoding.encode(text)
    total_tokens = len(encoded_text)

    step = max(int(chunk_token_size * (1 - overlap_ratio)), 1)
    overlap_tokens = chunk_token_size - step

    output = []
    token_pointer = 0

    while token_pointer < total_tokens:
        upper_token_pointer = min(token_pointer + chunk_token_size, total_tokens)

        # Tail window that adds no new tokens (already covered by the overlap)
        if token_pointer > 0 and upper_token_pointer - token_pointer <= overlap_tokens:
            break

        decoded_text = encoding.decode(encoded_text[token_pointer:upper_token_pointer])
        output.append({
            "text": decoded_text,
            "token_start": token_pointer,
            "token_count": upper_token_pointer - token_pointer,
        })

        token_pointer += step

    return output


Chunk by chapter of book 1 only, create `jsonl` file

In [15]:
chunk_text_output = []
def generate_chunk_id(chapter_num, idx)->str:
    return f"hp1-ch{chapter_num:02d}-{idx:04d}"

for _, text_chapter in enumerate(text_data):
    chapter_num = _ + 1
    chunked_texts = chunk_text_by_sliding_window(text_chapter)
    chapter_details = [
        i for i in index 
        if i["book_num"] == 1 and i["chapter_num"] == chapter_num
    ][0]

    chunk_id_count = 1
    for segment_details in chunked_texts:
        chunk_id = generate_chunk_id(
            chapter_num=chapter_num, idx=chunk_id_count
        )
        chunk_details = chapter_details | {
            "chunk_id": chunk_id,
            "token_start": segment_details.get("token_start", None),
            "token_count": segment_details.get("token_count", None),
            "text": segment_details.get("text", None)
        }
        chunk_text_output.append(chunk_details)
            
        chunk_id_count += 1


In [16]:
with open(DATA / "harrypotter-chunks.jsonl", "w", encoding="utf-8") as f:
    for chunk in chunk_text_output:
        # JSONL: only 1 object 1 line
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")


## Embedding into VectorDB

Using AWS Bedrock's `amazon.titan-embed-text-v2:0` embedding model. Store vectors into local `chromadb`

## Setup ChromaDB

In [17]:
chroma_client = chromadb.PersistentClient(path=str(LABS / ".chroma"))

# Define collection metadata

INDEX_CONFIG = {
    "embedding_model": "amazon.titan-embed-text-v2:0",
    "dimensions": 1024,
    "normalize": True,
    "chunk_size": 512,
    "overlap_ratio": 0.15,
    "tokenizer": "o200k_base",
    "chunking": "sliding_window",
}

# Utility function to check if extra embedding is performed other than Bedrock required

class BedrockOnly:
    """Sentinel embedding function: forces every vector to come from Bedrock."""

    def __call__(self, input):
        raise RuntimeError("Chroma must not embed; pass embeddings= from Bedrock.")

    def embed_query(self, input):
        raise RuntimeError("Use query_embeddings= with Bedrock vectors.")

    def name(self):
        return "bedrock_only"


# Get or create the vector collection. Embeddings always come from Bedrock.

def get_collection(client, name, config=INDEX_CONFIG):
    client.get_or_create_collection(
        name=name,
        embedding_function=BedrockOnly(),
        configuration={"hnsw": {"space": "cosine"}},   # distance = 1 - cos_sim
        metadata=config,                               # repro stamp, frozen at creation
    )
    return client.get_collection(name, embedding_function=BedrockOnly())


# Create collection

COLLECTION_NAME = "hp1-c512-ov15"

collection = get_collection(chroma_client, name=COLLECTION_NAME)

In [18]:
collection.count()

249

## Using Bedrock to embed text

Pass all chunks into embed to process in batch. In production, this should be done with multi-threading and avoid sync processing.

In [19]:
# Standard json library cannot load jsonl file directly
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

In [20]:
chunked_text_list = read_jsonl(DATA / "harrypotter-chunks.jsonl")
preview = chunked_text_list[0]
print("Characters in 1st chunk: ", len(preview.get("text", None)))

Characters in 1st chunk:  2144


In [21]:
chunks_only_list = [t.get("text", "") for t in chunked_text_list]
# Batch embedding
embedded_text_list = bedrock_client.embed(chunks_only_list)

In [22]:
embedded_text_list[0]

array([-0.02467984,  0.02871553,  0.04159872, ..., -0.05153274,
       -0.04842836, -0.04439266], shape=(1024,), dtype=float32)

## Validate embedding

In [23]:
# chunk_id -> ids, text -> documents, others -> metadata

METADATA_KEYS = (
    "book_num",
    "book_name",
    "chapter_num",
    "chapter_name",
    "chapter_start_page_num",
    "chapter_end_page_num",
    "token_start",
    "token_count", 
)


def validate_batch(chunks, vectors, dimensions=INDEX_CONFIG["dimensions"], metadata_keys=METADATA_KEYS):
    """Validate size/dtype of vector and metadata"""
    # Vector size
    expected = (len(chunks), dimensions)
    assert vectors.ndim == 2, f"expected 2-D embeddings, got {vectors.ndim}-D"
    assert vectors.shape == expected, f"expected {expected}, got {vectors.shape}"
    # Vector type
    assert np.isfinite(vectors).all(), "embeddings contain NaN/Inf"
    # Metadata keys
    required = set(metadata_keys)
    incomplete = [c["chunk_id"] for c in chunks if not required <= c.keys()]
    assert not incomplete, f"chunks missing metadata keys: {incomplete[:5]}"
    dropped = [c["chunk_id"] for c in chunks if any(c[k] is None for k in metadata_keys)]
    assert not dropped, f"None metadata would be silently dropped: {dropped[:5]}"


chunks = chunked_text_list
vectors = embedded_text_list
validate_batch(chunks, vectors)

## Push to chroma collection

In [24]:
collection.upsert(
    ids=[c["chunk_id"] for c in chunks],
    embeddings=vectors,
    documents=[c["text"] for c in chunks],
    metadatas=[{k: c[k] for k in METADATA_KEYS} for c in chunks],
)

collection.count()

249

## Test RAG workflow

### Embed question with Bedrock, not chroma

In [30]:
# One gold-set record as example

question_set = {
    "id": "hp1-016",
    "book": 1,
    "chapter": 16,
    "question": "How does Harry navigate through the room filled with the flying door opener?",
    "answer": "He rides a broomstick provided in the room to chase down and catch the heavy silver key.",
    "evidence": [{"chapter": 16, "contains": "look! Broomsticks!"}],
    "type": "factoid",
}

question_embedded = bedrock_client.embed(question_set["question"])
question_embedded

array([[-0.08496385,  0.04551635, -0.09042582, ...,  0.00508266,
        -0.00800329,  0.07616403]], shape=(1, 1024), dtype=float32)

### Query chroma collection and return top_k results

In [26]:
documents_embedded = collection.query(
    query_embeddings=question_embedded,
    n_results=5,
    include=["documents", "metadatas", "distances"]
)
[len(d) for d in documents_embedded["documents"][0]]

[2155, 2002, 2254, 2291, 2176]

In [27]:
# Zip results and metadata into single dict (rank is 1-based, for reporting/MRR)

top_k_documents = enumerate(zip(
    documents_embedded["ids"][0],
    documents_embedded["documents"][0],
    documents_embedded["metadatas"][0],
    documents_embedded["distances"][0],
), start=1)

top_k_documents = [
    {"chunk_id": chunk_id, "rank": rank, "distance": distance, "text": document, **meta}
        for rank, (chunk_id, document, meta, distance) in top_k_documents
]

### Build the base prompt format

In [28]:
REFUSAL = "I cannot answer that from the provided excerpts."

SYSTEM_TEMPLATE = """You answer questions about the book "{book_name}" using ONLY the excerpts provided in the user's message.

Rules:
1. Base every statement on the provided excerpts. Never use knowledge from outside them, even if you are confident about it.
2. Some excerpts may be irrelevant to the question. Ignore them. Prefer the excerpt that addresses the question most directly.
3. If the excerpts do not contain the answer, reply with exactly this sentence and nothing else:
{refusal}
4. When an excerpt states a concrete action, name it explicitly - the object, the place or the spell - rather than describing the character's general ability.
5. Answer in 1-3 sentences unless the question requires more.
6. After the answer, list the id attribute of each excerpt you relied on, one per line, no other text.
"""

# The excerpt label IS the citation, so the model can copy it verbatim. The chapter
# title lives in a separate attribute on purpose: putting it in the label made the
# model cite "[Book 1, Chapter 16 - Chapter 16 - Through the Trapdoor]".

EXCERPT_TEMPLATE = """<excerpt id="[Book {book_num}, Chapter {chapter_num}]" chapter="{chapter_name}">
{text}
</excerpt>"""

BOOK_NAME = index[0]["book_name"]


def build_prompt(question, records):
    excerpts = "\n\n".join(
        EXCERPT_TEMPLATE.format(
            book_num=r["book_num"], chapter_num=r["chapter_num"],
            chapter_name=r["chapter_name"], text=r["text"],
        )
        for r in records
    )
    system = SYSTEM_TEMPLATE.format(book_name=BOOK_NAME, refusal=REFUSAL)
    prompt = f"<excerpts>\n{excerpts}\n</excerpts>\n\nQuestion: {question}\n\nAnswer:"
    return system, prompt

### Prompt LLM and return result

In [29]:
system, prompt = build_prompt(question_set["question"], top_k_documents)

answer = bedrock_client.chat(
    prompt,
    system=system,
    temperature=0.0,
    max_tokens=1024
)

print(f"""\
Question:
  {question_set["question"]}
Answer:
  {answer}
Expected ({question_set["id"]}, {question_set["type"]}):
  {question_set["answer"]}
""")

Question:
  How does Harry navigate through the room filled with the flying door opener?
Answer:
  Harry navigates through the room filled with flying keys by using broomsticks to fly and catch one of the keys. He spots a large silver key with a bent wing and directs Ron and Hermione to help him catch it. Harry catches the key by pinning it against the wall with his hand.

[Book 1, Chapter 16]
Expected (hp1-016, factoid):
  He rides a broomstick provided in the room to chase down and catch the heavy silver key.



## First Evaluation

The pipeline is completed with a very simple chunking strategy (fixed size with overlap) and RAG is successful with the in-context document included in top 5 document. 

However, the testing question doesn't have jargons in magic world and the expected answer is generic enough. On dense retrieval level it is ok, therefore the generation is properly in place.

## Try Sparse retrieval as comparison

In [32]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

# Lexical index over the exact same chunks that were embedded
bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])

### Contrast pair: one fact, two phrasings

Both questions ask about the same passage, where Hermione taps the lock and
whispers **"Alohomora!"** - yet rewording the question flips which retriever finds it.

In [33]:
# Same fact, two phrasings - the winner flips with the wording
CONTRAST_PAIR = [
    {
        "id": "hp1-009a",
        "book": 1,
        "chapter": 9,
        "question": "What exact incantation is spoken to open the locked door on the third floor?",
        "answer": "Alohomora",
        "evidence": [{"chapter": 9, "contains": "“Alohomora!” The lock clicked and the door swung open"}],
        "type": "factoid",
    },
    {
        "id": "hp1-009b",
        "book": 1,
        "chapter": 9,
        "question": "What spell did Hermione use to open the locked door?",
        "answer": "Alohomora",
        "evidence": [{"chapter": 9, "contains": "“Alohomora!” The lock clicked and the door swung open"}],
        "type": "factoid",
    },
]

# The evidence snippet is also how we know which chunk counts as correct
GOLD_CHUNK = next(c["chunk_id"] for c in chunks if CONTRAST_PAIR[0]["evidence"][0]["contains"] in c["text"])


def compare_retrievers(question, top_k=5):
    """Run one question through dense and sparse retrieval, side by side."""
    dense = collection.query(
        query_embeddings=bedrock_client.embed(question),
        n_results=top_k,
        include=["distances"],
    )["ids"][0]

    scores = bm25.get_scores(tokenize(question))
    sparse = [chunks[i]["chunk_id"] for i in scores.argsort()[::-1][:top_k]]

    print(f"\nQ: {question}")
    for name, hits in (("dense ", dense), ("sparse", sparse)):
        print(f"  {name}: " + "  ".join(f"{c}*" if c == GOLD_CHUNK else c for c in hits))
    print(f"  (* = {GOLD_CHUNK})")


for record in CONTRAST_PAIR:
    compare_retrievers(record["question"])


Q: What exact incantation is spoken to open the locked door on the third floor?
  dense : hp1-ch09-0014*  hp1-ch09-0015  hp1-ch16-0016  hp1-ch16-0011  hp1-ch16-0015
  sparse: hp1-ch09-0015  hp1-ch08-0001  hp1-ch09-0016  hp1-ch16-0011  hp1-ch09-0011
  (* = hp1-ch09-0014)

Q: What spell did Hermione use to open the locked door?
  dense : hp1-ch10-0010  hp1-ch16-0015  hp1-ch16-0016  hp1-ch16-0012  hp1-ch16-0014
  sparse: hp1-ch09-0016  hp1-ch16-0001  hp1-ch09-0014*  hp1-ch16-0018  hp1-ch10-0011
  (* = hp1-ch09-0014)


| Question | dense | sparse |
|---|---|---|
| "exact incantation ... on the third floor?" | rank 1 | not in top 5 |
| "What spell did Hermione use ...?" | not in top 5 | rank 3 |

The passage never says "third floor", so BM25 has no lexical path to it in the
first phrasing - dense gets there on meaning alone. In the second phrasing dense
is pulled towards chapter 16, the *other* locked door, where the same charm fails.

The two retrievers fail on different questions. That is the argument for fusing
their rankings into a hybrid rather than picking one.

### Send each retriever's chunks to the model

The retrieval ranking only matters if it changes the answer

Now each retriever's own top-5 is passed to Bedrock for the same question

A miss = simply refusal, thanks to prompt's guardrail

In [34]:
CHUNK_BY_ID = {c["chunk_id"]: c for c in chunks}


def to_records(chunk_ids):
    """Retrieved chunk ids -> the record shape build_prompt() expects."""
    return [
        {"rank": rank, "text": CHUNK_BY_ID[cid]["text"],
         **{k: CHUNK_BY_ID[cid][k] for k in METADATA_KEYS}}
        for rank, cid in enumerate(chunk_ids, start=1)
    ]


def answer_with(question, chunk_ids):
    """Prompt Bedrock using one retriever's chunks."""
    system, prompt = build_prompt(question, to_records(chunk_ids))
    return bedrock_client.chat(prompt, system=system, temperature=0.0, max_tokens=512)


for record in CONTRAST_PAIR:
    question = record["question"]
    dense_hits = collection.query(
        query_embeddings=bedrock_client.embed(question),
        n_results=5,
        include=["distances"],
    )["ids"][0]
    sparse_hits = [
        chunks[i]["chunk_id"]
        for i in bm25.get_scores(tokenize(question)).argsort()[::-1][:5]
    ]

    print(f"\nQ: {question}")
    print(f"  expected answer: {record['answer']}")
    for name, hits in (("dense ", dense_hits), ("sparse", sparse_hits)):
        print(f"  --- {name} context ---")
        print("  " + answer_with(question, hits).replace("\n", "\n  "))


Q: What exact incantation is spoken to open the locked door on the third floor?
  expected answer: Alohomora
  --- dense  context ---
  The incantation spoken to open the locked door on the third floor is "Alohomora."
  
  [Book 1, Chapter 9]
  --- sparse context ---
  I cannot answer that from the provided excerpts.
  
  The excerpts do not mention the exact incantation spoken to open the locked door on the third floor. They describe the danger of the three-headed dog and the students' fear, but do not provide the specific spell or incantation used to unlock the door.

Q: What spell did Hermione use to open the locked door?
  expected answer: Alohomora
  --- dense  context ---
  Hermione used the Alohomora Charm to try to open the locked door, but it did not work.
  
  [Book 1, Chapter 16]
  --- sparse context ---
  Hermione used the spell "Alohomora" to open the locked door.
  
  [Book 1, Chapter 9]


## Second Evaluation

As a result, neither retriever wins outright: the winner flips with the wording of
the question, and the flip reaches the answer.

| Question | Retriever | Evidence in top 5 | Answer |
|---|---|---|---|
| "exact incantation ... on the third floor?" | dense | yes (rank 1) | "Alohomora" |
| "exact incantation ... on the third floor?" | sparse | no | refused |
| "What spell did Hermione use ...?" | dense | no | "the Alohomora Charm ... did not work" |
| "What spell did Hermione use ...?" | sparse | yes (rank 3) | "Alohomora" |

Two things worth more than the scores:

- **A retrieval miss does not always look like one.** Sparse's miss produced a
  correct refusal; dense's miss produced a confident, well-cited answer about the
  *other* locked door. Both are retrieval failures, and only the second one reads
  like a wrong answer.
- **The refusal is not a generation failure.** It is the guardrail working - and it
  is only distinguishable from over-abstention because the retrieved chunks were
  kept alongside the answer.

So the next step is not to crown a winner but to fuse the two rankings (hybrid /
reciprocal rank fusion) and re-measure. That needs the gold set and a harness, not
two questions read by eye.

## Wrap-up

Book 1 end to end: PyMuPDF to chapter-aware text (17 chapters), 512-token
overlapping chunks (249 records), Titan Text v2 embeddings with a disk cache,
Chroma with cosine distance, dense and sparse retrieval, then grounded answers
from Bedrock Converse.

Three things this run established:

- **Retrieval and generation fail independently.** The same kind of miss shows up
  once as a refusal and once as a confident wrong answer, so scoring the two
  stages together hides which one broke.
- **The grounding rules hold.** Given context that does not contain the answer,
  the model declines instead of answering from what it already knows about the
  series - which matters, because the series is in its training data.
- **Neither retriever wins outright.** The winner flips with the wording of the
  question, which is the argument for fusing them rather than picking one.

Still open, by design: the gold set is two records and every score here was read
by eye. `Hit@k`, `MRR`, the LLM judge and the ablation table come next - and
because correctness alone can be satisfied by the model's own memory, every
result needs a closed-book and a placebo run to mean anything.